# Тяжёлые металлы на бывших складах пестицидов: что показывают данные XRF и что умеет модель

Демонстрационный блокнот пилотного проекта. Он показывает, как устроены расчёты стенда для металлов, на реальных
данных и без «чёрного ящика»: каждый график строится здесь же из выгрузки проекта.

**Что внутри**

1. Данные: 10 складов, 169 точек полевого рентгенофлуоресцентного анализа (XRF), 9 элементов.
2. Сравнение с нормативами: сколько точек превышает национальные и международные уровни.
3. Приоритет складов: какой склад загрязнён сильнее остальных.
4. Модель: где на складе искать очаг, и как это проверяется на складе, который модель не видела.
5. Сколько данных нужно, чтобы модель стала точнее.
6. Связь с участками СОЗ‑пестицидов.

**Как читать.** Код скрыт; кнопка «Показать код» вверху страницы раскрывает его. Все расчёты выполняются локально
из файла `tajikistan_soil_ml_export.gpkg`, без базы данных и без интернета.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
import mltj, demo_utils as U
from demo_utils import f, pct
U.setup()
sites, pts, cov = mltj.load()
wgs_sites, wgs_pts = sites.to_crs(4326), pts.to_crs(4326)
print(f"Складов: {len(sites)}, точек XRF: {len(pts)}, элементов: {len(mltj.ELEMENTS)}")

## 1. Данные

На каждом складе точки XRF расположены внутри контура склада и вокруг него. Для каждой точки известны концентрации
9 элементов (As, Co, Hg, Mo, Pb, Rb, Th, U, Zn), расстояние до контура склада, до воды и pH почвы.

In [ ]:
g = pts.groupby('site_name')
t = pd.DataFrame({
    'Склад': g.size().index,
    'Точек': g.size().values,
    'Внутри контура': g.inside.sum().astype(int).values,
    'Pb, медиана, мг/кг': g.Pb_mgkg.median().values,
    'Pb, максимум, мг/кг': g.Pb_mgkg.max().values,
}).sort_values('Pb, медиана, мг/кг', ascending=False)
U.table(t, nd=0)

b = wgs_pts.total_bounds; pad = .35
fig, ax = plt.subplots(figsize=(8.5, 5.6))
U.hillshade(ax, (b[0]-pad, b[1]-pad, b[2]+pad, b[3]+pad))
c = wgs_sites.geometry.centroid
med = g.Pb_mgkg.median().reindex(wgs_sites.name.values).values
sc = ax.scatter(c.x, c.y, s=90 + 3*med, c=med, cmap='YlOrRd', edgecolor='#333', lw=.8, zorder=3, vmin=0)
OFF = {'Турсунзаде': (-12, -16, 'right'), 'Шахринав': (0, 12, 'center'), 'Гиссар': (10, -14, 'left')}
for (x, y_), n in zip(zip(c.x, c.y), wgs_sites.name):
    dx, dy, ha = OFF.get(n, (9, 4, 'left'))
    ax.annotate(n, (x, y_), xytext=(dx, dy), textcoords='offset points', fontsize=9, zorder=4, ha=ha,
                bbox=dict(boxstyle='round,pad=.15', fc='white', ec='none', alpha=.75))
plt.colorbar(sc, ax=ax, shrink=.7, label='Pb, медиана по складу, мг/кг')
U.map_axes(ax); ax.set_title('10 складов с XRF-съёмкой (размер и цвет — медиана свинца)')
plt.show()

## 2. Сравнение с нормативами

Используются два уровня сравнения:

- **ПДК** по перечню №6229‑91 (СССР), на который ссылается заказчик: Pb 32, As 2, Hg 2,1 мг/кг; для цинка в перечне
  только подвижная форма, поэтому взята ОДК валового цинка для нейтральных суглинков (220 мг/кг; pH почв на складах
  7,2–7,7);
- **международные уровни для жилой застройки**: Pb 200 (EPA, 2025), As 12, Hg 6,6, Zn 250 мг/кг (CCME, Канада).

Важная оговорка: у портативного XRF предел обнаружения мышьяка и ртути близок к самим нормативам. Превышения по As и
Hg — сигнал для лабораторной проверки, а не окончательный вывод. Для свинца и цинка XRF надёжен.

In [ ]:
N = {'Pb': (32, 200), 'As': (2, 12), 'Hg': (2.1, 6.6), 'Zn': (220, 250)}
rows = []
for e, (pdk, intl) in N.items():
    v = pts[f'{e}_mgkg']
    rows.append({'Элемент': e, 'Медиана, мг/кг': v.median(), 'Максимум, мг/кг': v.max(),
                 'ПДК / ОДК, мг/кг': pdk, 'Выше ПДК': (v > pdk).mean(),
                 'Междунар. уровень, мг/кг': intl, 'Выше междунар. уровня': (v > intl).mean()})
r = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(r)); w = .38
ax.bar(x - w/2, 100*r['Выше ПДК'], w, color=U.ORANGE, label='выше ПДК / ОДК (СССР)')
ax.bar(x + w/2, 100*r['Выше междунар. уровня'], w, color=U.BLUE, label='выше международного уровня (жилая зона)')
for i, rr in r.iterrows():
    ax.text(i - w/2, 100*rr['Выше ПДК'] + 2, pct(rr['Выше ПДК']), ha='center', fontsize=9)
    ax.text(i + w/2, 100*rr['Выше междунар. уровня'] + 2, pct(rr['Выше междунар. уровня']), ha='center', fontsize=9)
ax.set_xticks(x, r['Элемент']); ax.set_ylim(0, 112); ax.set_ylabel('доля точек, %')
ax.set_title('Доля из 169 точек XRF выше нормативов'); ax.legend(loc='upper center', bbox_to_anchor=(.5, -.1), ncol=2, fontsize=9)
plt.show()

r2 = r.copy()
r2['Выше ПДК'] = r2['Выше ПДК'].map(pct); r2['Выше междунар. уровня'] = r2['Выше междунар. уровня'].map(pct)
U.table(r2, nd=1)

In [ ]:
pb = pts.Pb_mgkg
U.note(f"Свинец: {pct((pb > 32).mean())} точек выше ПДК 32 мг/кг, но только {int((pb > 200).sum())} из {len(pb)} — "
       f"выше международного уровня 200 мг/кг. Загрязнение свинцом распространённое, но умеренное; "
       f"мышьяк и ртуть требуют лабораторного подтверждения.")

## 3. Приоритет складов

Для каждой точки считается, во сколько раз её концентрации Pb, As, Hg и Zn выше типичных (медиан по всей съёмке),
в логарифмической шкале; затем — среднее по четырём элементам. Индекс склада — медиана по его точкам. Метод не зависит
от выбора норматива и сравнивает склады между собой.

In [ ]:
sub = ['Pb', 'As', 'Hg', 'Zn']
ex = pd.concat([np.log(pts[f'{e}_mgkg']) - np.log(pts[f'{e}_mgkg']).median() for e in sub], axis=1)
pts['composite'] = ex.mean(axis=1)
pr = pts.groupby('site_name').composite.median().sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4))
cols = [U.BLUE if i >= len(pr) - 3 else U.GREY for i in range(len(pr))]
ax.barh(pr.index, pr.values, color=cols)
ax.axvline(0, color='#555', lw=.8)
for i, v in enumerate(pr.values):
    ax.text(v + (.01 if v >= 0 else -.01), i, f(v, 2), va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlim(pr.min() - .14, pr.max() + .12)
ax.set_xlabel('индекс загрязнения склада (0 — типичный уровень по съёмке)')
ax.set_title('Приоритет складов по металлам: три первых выделены')
U.comma_axis(ax, 'x', 1)
plt.show()
same = list(pr.index[::-1]) == list(sites.sort_values('priority_rank').name)
U.note(f"Первые по приоритету: {', '.join(pr.index[::-1][:3])}. " + ("Порядок совпадает с рейтингом на стенде."
       if same else "Порядок отличается от рейтинга на стенде — проверьте версию выгрузки."))

## 4. Модель: где на складе искать очаг

**Задача модели.** Предсказать, в какой части склада свинца больше, чем обычно для этого склада. Целевая величина —
отклонение log(Pb) точки от медианы её склада: так модель учится положению очага, а не различиям между складами.

**Признаки точки:** расстояние до контура (внутри или снаружи), расстояние до центра склада, зона (внутри, до 50 м,
до 100 м, дальше), расстояние до воды, pH почвы.

**Алгоритм:** случайный лес (300 деревьев). **Проверка:** обучение на 9 складах, прогноз для десятого, который
модель не видела (leave‑one‑warehouse‑out); так по очереди для каждого склада.

In [ ]:
y = mltj.target(pts, 'Pb')
res = []
for name, feats in [('только положение точки относительно склада', mltj.BASE),
                    ('+ расстояние до воды и pH (модель стенда)', mltj.FEATURES)]:
    r2 = [mltj.lowo_r2(pts, y, feats, seed=s)[0] for s in range(5)]
    res.append({'Набор признаков': name, 'R², среднее': np.mean(r2), 'R², мин': np.min(r2), 'R², макс': np.max(r2)})
U.table(pd.DataFrame(res), nd=3)
P = np.mean([mltj.lowo_predict(pts, y, mltj.FEATURES, seed=s) for s in range(5)], axis=0)
pts['pred'] = P

R² около 0,16 означает, что модель объясняет примерно шестую часть различий внутри складов. Этого мало для
количественного прогноза концентраций, но достаточно, чтобы **подсказать, где брать пробы в первую очередь**.
Ниже — пример склада, которого модель не видела при обучении.

In [ ]:
site = pr.index[-1]   # склад с наивысшим приоритетом
S = pts[pts.site_name == site]; poly = sites.loc[S.site_id.iloc[0]].geometry
fig, axs = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True, sharey=True)
for ax, col, title, cmap in [(axs[0], S.Pb_mgkg, f'{site}: измеренный Pb, мг/кг', 'YlOrRd'),
                             (axs[1], S.pred.rank(pct=True), f'{site}: оценка модели (ранг внутри склада)', 'Blues')]:
    gx, gy = poly.exterior.xy
    ax.fill(gx, gy, color='#eee', ec='#555', lw=1.2, zorder=1)
    sc = ax.scatter(S.geometry.x, S.geometry.y, c=col, cmap=cmap, s=70, edgecolor='#333', lw=.6, zorder=3)
    plt.colorbar(sc, ax=ax, shrink=.75)
    ax.set_title(title, fontsize=10); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
k = int(np.ceil(len(S) * .25)); top = S.nlargest(k, 'pred')
axs[1].scatter(top.geometry.x, top.geometry.y, s=220, facecolor='none', edgecolor=U.ORANGE, lw=2, zorder=4,
               label='первая четверть по модели')
axs[1].legend(loc='upper center', bbox_to_anchor=(.5, -.02), fontsize=8)
plt.suptitle('Модель обучена на остальных 9 складах', fontsize=10, color='#555'); plt.show()

In [ ]:
rows = []
for s, gg in pts.groupby('site_name'):
    k = int(np.ceil(len(gg) * .25))
    top_pred, top_act = set(gg.nlargest(k, 'pred').index), set(gg.nlargest(k, 'Pb_mgkg').index)
    rows.append({'Склад': s, 'Точек': len(gg), 'Первая четверть, точек': k,
                 'Доля самых загрязнённых точек, найденных моделью': len(top_pred & top_act) / k})
cap = pd.DataFrame(rows).sort_values('Доля самых загрязнённых точек, найденных моделью', ascending=False)
m = cap['Доля самых загрязнённых точек, найденных моделью'].mean()

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(cap['Склад'], 100*cap['Доля самых загрязнённых точек, найденных моделью'], color=U.BLUE)
ax.axhline(25, color=U.GREY, ls='--', lw=1.5); ax.text(9.4, 26.5, 'случайный выбор: 25 %', ha='right', color='#555', fontsize=9)
ax.axhline(100*m, color=U.BLUE, ls=':', lw=1.5); ax.text(9.4, 100*m + 1.5, f'в среднем: {pct(m)}', ha='right', color=U.BLUE, fontsize=9)
ax.set_ylabel('%'); ax.set_ylim(0, 100); plt.xticks(rotation=35, ha='right')
ax.set_title('Сколько из 25 % самых загрязнённых точек попадает в первую четверть по модели')
plt.show()
U.note(f"Если брать пробы сначала в четверти точек, отмеченных моделью, в среднем находится {pct(m)} самых загрязнённых "
       f"точек склада; при случайном выборе — 25 %. Выигрыш реальный, но умеренный, и на разных складах разный.")

## 5. Сколько данных нужно

Модель обучается на 3, 5, 7 и 9 случайно выбранных складах и каждый раз проверяется на складе вне обучения.
Если качество растёт с числом складов, ограничение — объём данных, а не метод.

In [ ]:
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
F = mltj.FEATURES; ids = list(pts.site_id.unique()); lc = {}
for k in (3, 5, 7, 9):
    vals = []
    for rep in range(8):
        rnd = random.Random(rep * 100 + k); pred = np.zeros(len(y))
        for te_s in ids:
            tr_s = rnd.sample([s for s in ids if s != te_s], k)
            tr, te = pts.site_id.isin(tr_s).values, (pts.site_id == te_s).values
            pred[te] = RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=rep).fit(
                pts.loc[tr, F].values, y[tr]).predict(pts.loc[te, F].values)
        vals.append(r2_score(y, pred))
    lc[k] = vals
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ks = list(lc); mu = [np.mean(lc[k]) for k in ks]
ax.plot(ks, mu, 'o-', color=U.BLUE, lw=2)
ax.fill_between(ks, [np.percentile(lc[k], 10) for k in ks], [np.percentile(lc[k], 90) for k in ks], color=U.BLUE, alpha=.15)
for k, v in zip(ks, mu): ax.annotate(f(v, 2), (k, v), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)
ax.axhline(0, color='#555', lw=.8)
ax.set_xticks(ks); ax.set_xlabel('складов в обучении'); ax.set_ylabel('R² на складе вне обучения')
ax.set_title('Качество модели в зависимости от числа складов в обучении')
import matplotlib.ticker as mt; ax.yaxis.set_major_locator(mt.MultipleLocator(.05)); U.comma_axis(ax, 'y', 2)
plt.show()
U.note(f"При 3 складах в обучении R² = {f(mu[0], 2)}, при 9 — {f(mu[-1], 2)}. Рост неравномерный, разброс между "
       f"повторами большой (полоса на графике): десяти складов мало, чтобы оценить модель уверенно. Новые склады нужны "
       f"и для качества, и для надёжности оценки.")

## 6. Связь с участками СОЗ‑пестицидов

9 из 10 складов XRF совпадают с участками СОЗ на портале chemicals.tj. На шести из них есть анализы на ДДТ и ГХЦГ;
это позволяет сравнить загрязнение металлами и пестицидами на одних площадках.

In [ ]:
lk = U.DATA / 'source_data/chemicals_tj/pop_layer/metals_link'
sl = pd.read_csv(lk / 'site_level.csv', encoding='utf-8-sig')
summ = json.loads((lk / 'summary.json').read_text(encoding='utf-8'))
t = pd.DataFrame({'Склад XRF': sl.xrf_site, 'Участок СОЗ': sl.tj_code, 'Приоритет по металлам': sl.metals_priority_rank,
                  'Pb, медиана XRF, мг/кг': sl.xrf_Pb_median, 'Проб СОЗ': sl.pop_samples,
                  'ДДТ+ГХЦГ, максимум, мг/кг': sl.pop_burden_max_mgkg}).sort_values('Приоритет по металлам')
U.table(t, nd=1)
s_ = summ['site_level_spearman']
U.note(f"Ранговая корреляция приоритета по металлам и загрязнения пестицидами по {s_['n']} площадкам: "
       f"{f(s_['rho'], 2)} (p = {f(s_['p'], 2)}) — связь на уровне площадок не подтверждается.")

На уровне площадок связь слабая, но на трёх складах самые загрязнённые пробы пестицидов лежат рядом с точками
максимума свинца (Дж. Балхи, Дусти, Кушониён). Это основание для совместного отбора: XRF и лабораторные пробы на
пестициды в одних точках. Если связь подтвердится, дешёвая полевая XRF‑съёмка будет подсказывать, где брать дорогие
лабораторные пробы.

## Выводы

- Свинец выше ПДК почти в половине точек, но выше международного уровня для жилой зоны — единично; мышьяк и ртуть
  требуют лабораторной проверки из‑за предела обнаружения XRF.
- Приоритет складов считается прозрачно и не зависит от выбора норматива.
- Модель внутри склада даёт умеренный, но проверенный на «чужих» складах выигрыш при выборе мест отбора.
- Главное ограничение — 10 складов: при 9 складах в обучении качество выше, чем при меньшем числе, но разброс
  велик. Каждый новый склад повышает и качество, и надёжность оценки.